# 19 — Channel factorial: belief accuracy vs closed-loop profit

One **joint shard** per `(seed, ObsChannels)` runs a single closed-loop `act()` episode and records **mean-f MAE**, **8-bin distribution MAE**, and **profit** from the same scored days.

Canonical grid: `(upc|gsin) × (waste on|off) × (none|pack_date|temperature_history)` — 12 cells.

Budget target: ~20 min wall / 2 CPU-hr on Modal (probe → plan → run).

In [1]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path
from typing import Literal

import pandas as pd

REPO_ROOT = Path.cwd() if (Path.cwd() / "src" / "blueberries_voi").is_dir() else Path.cwd().parent
DATA_DIR = REPO_ROOT / "experiments" / "data"
FIG_DIR = REPO_ROOT / "figures" / "channel_joint"
OUT_JSON = DATA_DIR / "nb19_joint_rows.json"

_wheel_dir = REPO_ROOT / "dist" / "wheel"
os.environ.setdefault("BLUEBERRIES_VOI_BACKEND", "rust")
if _wheel_dir.is_dir():
    os.environ["BLUEBERRIES_VOI_WHEEL"] = str(_wheel_dir)

from blueberries_voi.experiments.batch_budget import assert_within_budget, plan_channel_joint_budget
from blueberries_voi.experiments.channel_factorial_viz import save_nb19_figures
from blueberries_voi.experiments.channel_joint import (
    all_obs_channels_product,
    channel_joint_job_grid,
    run_seed_channel_joint,
)
from blueberries_voi.experiments.modal_dispatch import run_batch
from blueberries_voi.experiments.voi_profit import load_damped_sw_bo_params
from blueberries_voi.filter.types import channels_for_preset

BATCH_MODE: Literal["modal", "local"] = "modal"
SMOKE = False
ACCURACY_METRIC: Literal["mean_f", "distribution"] = "mean_f"

PROBE_SEED = 42
PROBE_CHANNEL = channels_for_preset("P0")
CHANNELS = all_obs_channels_product()
CANDIDATE_SEEDS = (42, 7, 99, 101, 2024, 31415)
BO_JSON = REPO_ROOT / "outputs" / "damped_sw_alpha_bo.json"
CONTROLLER_ALPHA, CONTROLLER_RHO = load_damped_sw_bo_params(BO_JSON)
print(f"controller damped_sw α={CONTROLLER_ALPHA:.4f} ρ={CONTROLLER_RHO:.4f}")


controller damped_sw α=0.8132 ρ=1.0000


## Probe — one shard wall time

In [2]:
probe_t0 = time.perf_counter()
probe_row = run_seed_channel_joint(
    PROBE_SEED,
    PROBE_CHANNEL,
    n_burn=2,
    n_score=10,
    controller_alpha=CONTROLLER_ALPHA,
    controller_rho=CONTROLLER_RHO,
    bo_json_path=BO_JSON,
)
probe_elapsed_s = time.perf_counter() - probe_t0
print(f"probe elapsed_s={probe_elapsed_s:.1f} mae_f={probe_row['mae_f']:.4f} profit={probe_row['profit']:.2f}")

probe elapsed_s=161.7 mae_f=0.0609 profit=318.50


## Plan — greedy seeds then bump `n_score` under Modal budget

In [3]:
plan = plan_channel_joint_budget(probe_elapsed_s, max_seeds=len(CANDIDATE_SEEDS))
assert_within_budget(plan)
SEEDS = tuple(CANDIDATE_SEEDS[:plan.n_seeds])
N_BURN = plan.n_burn
N_SCORE = plan.n_score
print(plan.as_dict())
print(f"grid={len(SEEDS)} seeds × {len(CHANNELS)} channels = {len(SEEDS) * len(CHANNELS)} shards")

{'n_seeds': 3, 'n_score': 30, 'n_burn': 2, 'n_channels': 12, 'shard_count': 36, 't_shard_s': 161.6593964260028, 'est_wall_s': 323.3187928520056, 'est_cpu_hr': 1.6165939642600278}
grid=3 seeds × 12 channels = 36 shards


## Run — Modal or local batch

In [4]:
run_t0 = time.perf_counter()
if OUT_JSON.is_file() and not SMOKE:
    rows = json.loads(OUT_JSON.read_text(encoding="utf-8"))
    run_wall_s = 0.0
    print(f"loaded {len(rows)} rows from {OUT_JSON.relative_to(REPO_ROOT)}")
else:
    rows = run_batch(
        "channel_joint",
        BATCH_MODE,
        smoke=SMOKE,
        seeds=SEEDS,
        channels=CHANNELS,
        n_burn=N_BURN,
        n_score=N_SCORE,
        n_rollout_paths=0,
        out_path=OUT_JSON,
        controller_alpha=CONTROLLER_ALPHA,
        controller_rho=CONTROLLER_RHO,
        bo_json_path=BO_JSON,
    )
    run_wall_s = time.perf_counter() - run_t0
    print(f"run wall_s={run_wall_s:.1f} rows={len(rows)}")
df = pd.DataFrame(rows)
df.head()

modal batch:   0%|          | 0/36 [00:00<?, ?shard/s]

[2026-08-27T15:38:01Z] modal batch collected 36 shards


run wall_s=67.3 rows=36


,seed,key,profit,stockout,code_type,scan_waste,delivery_history,preset,waste_total,waste,delivery,mae_f,mae_dist,n_burn,n_score,n_live_days
0,7,code=gsin|waste=0|hist=none,1064.5,52,gsin,False,none,custom,105,off,none,0.105690,0.071004,2,30,30
1,42,code=gsin|waste=0|hist=none,1305.5,28,gsin,False,none,custom,51,off,none,0.087235,0.063515,2,30,30
2,99,code=gsin|waste=0|hist=none,1019.0,49,gsin,False,none,custom,104,off,none,0.119276,0.079815,2,30,30
3,7,code=gsin|waste=0|hist=pack_date,1043.5,55,gsin,False,pack_date,custom,109,off,pack_date,0.070373,0.053492,2,30,30
4,42,code=gsin|waste=0|hist=pack_date,1332.0,17,gsin,False,pack_date,custom,70,off,pack_date,0.048562,0.043234,2,30,30


## Audit — shard coverage and CPU estimate

In [5]:
expected = len(channel_joint_job_grid(SEEDS, CHANNELS))
assert len(rows) == expected, (len(rows), expected)
audit_path = DATA_DIR / "nb19_run_audit.json"
if audit_path.is_file():
    audit = json.loads(audit_path.read_text(encoding="utf-8"))
    audit["accuracy_metric"] = ACCURACY_METRIC
else:
    est_cpu_hr = (len(rows) * probe_elapsed_s) / 3600.0
    audit = {
        "shards": len(rows),
        "seeds": list(SEEDS),
        "n_score": N_SCORE,
        "n_burn": N_BURN,
        "probe_elapsed_s": probe_elapsed_s,
        "run_wall_s": run_wall_s,
        "est_cpu_hr": est_cpu_hr,
        "accuracy_metric": ACCURACY_METRIC,
    }
print(json.dumps(audit, indent=2))
agg = df.groupby(["code_type", "waste", "delivery"], observed=True).agg(
    mae_f=("mae_f", "mean"),
    mae_dist=("mae_dist", "mean"),
    profit=("profit", "mean"),
)
agg

{
  "shards": 36,
  "seeds": [
    42,
    7,
    99
  ],
  "n_score": 30,
  "n_burn": 2,
  "probe_elapsed_s": 161.6593964260028,
  "run_wall_s": 67.31573923199903,
  "est_cpu_hr": 1.6165939642600278,
  "accuracy_metric": "mean_f"
}


mae_f  mae_dist       profit
code_type waste delivery                                            
gsin      off   none                 0.104067  0.071445  1129.666667
                pack_date            0.064967  0.050938  1132.000000
                temperature_history  0.060846  0.045589  1150.500000
          on    none                 0.104968  0.072800  1129.666667
                pack_date            0.065751  0.051085  1132.000000
                temperature_history  0.060441  0.046250  1150.500000
upc       off   none                 0.058936  0.058035  1124.333333
                pack_date            0.063622  0.055491  1135.666667
                temperature_history  0.064076  0.059809  1147.166667
          on    none                 0.055748  0.057395  1136.000000
                pack_date            0.063203  0.056025  1167.166667
                temperature_history  0.064244  0.059470  1149.333333

## Plots

In [6]:
acc_col = "mae_f" if ACCURACY_METRIC == "mean_f" else "mae_dist"
written = save_nb19_figures(rows, FIG_DIR, accuracy_column=acc_col)
for path in written:
    print(path.relative_to(REPO_ROOT))

/home/oliver/blog/blueberries-voi/.worktrees/T-163-notebook-pipeline/src/blueberries_voi/experiments/channel_factorial_viz.py:77: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


figures/channel_joint/channel_factorial_heatmap_mae_f.png
figures/channel_joint/profit_vs_mae_f.png
figures/channel_joint/parallel_coords_mae_f.png
